# **Modelado Predictivo de la Cibercriminalidad en España (2022-2024)**

Este cuaderno constituye el **Notebook 02** del proyecto, centrado en el **Análisis Exploratorio de Datos (EDA) No Gráfico** y la arquitectura de limpieza.

# **1. Análisis Exploratorio de Datos (EDA) No Gráfico**

En esta fase se realiza una primera inspección técnica de los datos. El objetivo no es la visualización, sino la **validación estadística y estructural** de la información antes de proceder a análisis más complejos.

Este análisis preliminar permitirá:
1. **Verificar la integridad:** Detectar posibles duplicados o registros inconsistentes tras la fase de limpieza.
2. **Inspección de Estructuras:** Evaluar mediante métodos como `.head()`, `.info()` y `.describe()` la naturaleza de cada columna.
3. **Identificación de Ajustes:** Determinar si se requieren transformaciones adicionales, recodificaciones o cruces de variables (*merging*) previos a la representación gráfica.

**Datasets de Trabajo**  
Para este análisis, partimos exclusivamente de los archivos extraidos del SEC e INE ubicados en la ruta `01_data`:

* `hechos_22_24.csv`  
* `ine_22_24.csv`

# Leyendas CSV

**Leyenda — `hechos_22_24.csv`**

| Columna | Descripción |
|--------|-------------|
| **Provincias** | Provincia o agregado territorial (*Total Nacional*). |
| **Grupo penal** | Categoría delictiva según clasificación SEC. |
| **periodo** | Año natural del registro (2022–2024). |
| **Total** | Número total de hechos conocidos registrados (entero). |

---

**Leyenda — `ine_22_24.csv`**

| Columna | Descripción |
|--------|-------------|
| **Provincias** | Provincia o agregado territorial (*Total Nacional*). |
| **Total** | Población residente a 1 de enero, ajustada al año natural (entero). |
| **periodo** | Año natural correspondiente (2022–2024). |

In [1]:
#@title 1.1 Ingesta y configuración del entorno
#En esta sección se prepara el entorno de trabajo mediante la importación de
#librerías y la configuración inicial necesaria para el análisis.

# Importar librerías necesarias

# El orden sigue una jerarquía de "de fuera hacia dentro":
# Sistema -> Análisis -> Visualización.

# --- Gestión de archivos, sistema y Carga de Datos (Local/GitHub) ---
import os                        # Navegación por el sistema de archivos
import requests                  # Peticiones HTTP (descarga de archivos GeoJSON/datos desde URL)

# --- Manipulación y limpieza de datos ---
import pandas as pd              # Estructuras de datos y análisis estadístico
import numpy as np               # Operaciones matemáticas y matrices

# --- Normalización de texto ---
import unicodedata               # Manejo de caracteres Unicode (quitar tildes, acentos, normalizar texto)
# ¿Para qué sirve unicodedata?
# → Permite transformar caracteres acentuados (á, é, í, ó, ú, ñ) en su versión simple (a, e, i, o, u, n)
#   Esto es clave cuando queremos comparar textos sin que los acentos generen categorías distintas.
#   Ejemplo: "Ávila" y "Avila" pasan a ser equivalentes.
import re                         # Expresiones regulares (limpieza y corte de texto avanzado)
# ¿Para qué sirve re?
# → Permite buscar patrones complejos y "limpiar" el texto de forma quirúrgica.
#   En este proyecto es clave para cortar nombres de provincias (ej. Araba/Álava -> Araba)
#   y normalizar los grupos de edad (ej. "De 18 a 25 años" -> "18_25").

# --- Visualización de datos ---
import seaborn as sns             # Gráficos estadísticos estáticos (ideales para la memoria escrita)
import matplotlib.pyplot as plt   # Motor base de gráficos y personalización
import matplotlib.ticker as mtick  # Formateo de ejes (puntos de millar, porcentajes)
import plotly.express as px       # Visualizaciones interactivas y mapas rápidos (Choropleth)
import json                       # Manipulación de archivos GeoJSON (necesario para las coordenadas del mapa)
import plotly.graph_objects as go # Control total sobre figuras (capas, mapas complejos y botones)
from plotly.subplots import make_subplots
# ¿Para qué sirve make_subplots?
# → Es el "arquitecto" de las visualizaciones avanzadas.
#   Permite crear gráficos con doble eje Y (esencial para comparar Hechos vs Víctimas)
#   o cuadrículas de varios gráficos en una sola imagen, manteniendo escalas independientes.

# --- Framework de Machine Learning y Pipeline ---
from sklearn.base import BaseEstimator, TransformerMixin
# ¿Para qué sirve esto?
# → Permite crear "transformadores" personalizados que se integran con Scikit-Learn.
#   Es lo que usare para que la limpieza de provincias sea automática y reutilizable.

# --- Herramientas de Análisis Exploratorio Automático (EDA) ---
# Se dejan comentadas para activar según necesidad de profundidad
# !pip install ydata-profiling
# from ydata_profiling import ProfileReport

print("Entorno de trabajo listo. Librerías cargadas correctamente.")

Entorno de trabajo listo. Librerías cargadas correctamente.


In [2]:
#@title 1.2. Carga, preparación de los datos y Head Revisión Inicial de los Datasets
#En esta sección se realiza la ingesta automatizada de los archivos CSV,
#estableciendo las estructuras de datos base que serán utilizadas en el análisis posterior.


# Nota: La librería os ya fueron importadas en la celda anterior

# 1. Definición de rutas (Paths)
# Usamos un nombre de variable consistente
path_data = os.path.join("..", "01_Data")

# 2. Carga de los datasets
try:
    # Asegúrar usar 'path_data' que es la variable definida arriba
    df_hechos = pd.read_csv(os.path.join(path_data, 'hechos_22_24.csv'), sep=';')
    df_poblacion = pd.read_csv(os.path.join(path_data, 'ine_22_24.csv'), sep=';')

    print(f"✅ Éxito: Archivos cargados correctamente desde {path_data}")

# 3. Visualización Inicial (Head)
    print("--- Muestra de HECHOS (2022-2024) ---")
    display(df_hechos.head())

    print("\n--- Muestra de POBLACIÓN INE (2022-2024) ---")
    display(df_poblacion.head())

except Exception as e:
    print(f"❌ Error al cargar los archivos: {e}")
    print(f"Verifica que los archivos CSV están en la carpeta: {path_data}")

✅ Éxito: Archivos cargados correctamente desde ..\01_Data
--- Muestra de HECHOS (2022-2024) ---


,Provincias,Grupo penal,periodo,Total
0,Total Nacional,ACCESO E INTERCEPTACIÓN ILÍCITA,2024,9773
1,Total Nacional,ACCESO E INTERCEPTACIÓN ILÍCITA,2023,7367
2,Total Nacional,ACCESO E INTERCEPTACIÓN ILÍCITA,2022,5578
3,Total Nacional,AMENAZAS Y COACCIONES,2024,17738
4,Total Nacional,AMENAZAS Y COACCIONES,2023,17472



--- Muestra de POBLACIÓN INE (2022-2024) ---


,Provincias,Total,periodo
0,Total Nacional,49128297,2024
1,Total Nacional,48619695,2023
2,Total Nacional,48085361,2022
3,02 Albacete,390751,2024
4,02 Albacete,389070,2023


# **2. Revisión inicial de los datos**

Se realiza una revisión preliminar de los datasets para validar su estructura, tipos de datos y detectar posibles problemas de calidad como valores nulos, duplicados o inconsistencias.

In [3]:
#@title Info Datasets

print("--- INFORMACIÓN: HECHOS ---")
df_hechos.info()

print("\n--- INFORMACIÓN: POBLACIÓN (INE) ---")
df_poblacion.info()


--- INFORMACIÓN: HECHOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1485 entries, 0 to 1484
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Provincias   1485 non-null   object
 1   Grupo penal  1485 non-null   object
 2   periodo      1485 non-null   int64 
 3   Total        1485 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 46.5+ KB

--- INFORMACIÓN: POBLACIÓN (INE) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159 entries, 0 to 158
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Provincias  159 non-null    object
 1   Total       159 non-null    int64 
 2   periodo     159 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 3.9+ KB


In [4]:
#@title Verificación de Duplicados y Estadística Descritiva

# --- DUPLICADOS ---
print("--- DUPLICADOS ---")
print(f"Hechos: {df_hechos.duplicated().sum()}")

# Mostrar SOLO las primeras filas duplicadas de Población (no todo el DataFrame)
dup_poblacion = df_poblacion[df_poblacion.duplicated(subset=['Provincias', 'periodo'], keep=False)]
print(f"Población (n duplicados): {dup_poblacion.shape[0]}")

# --- ESTADÍSTICA ---
print("\n--- ESTADÍSTICA: HECHOS ---")
display(df_hechos.describe().T)

print("\n--- ESTADÍSTICA: POBLACIÓN ---")
display(df_poblacion.describe().T)



--- DUPLICADOS ---
Hechos: 0
Población (n duplicados): 0

--- ESTADÍSTICA: HECHOS ---


,count,mean,std,min,25%,50%,75%,max
periodo,1485.0,2023.000000,0.816772,2022.0,2022.0,2023.0,2024.0,2024.0
Total,1485.0,3533.462626,27103.636896,0.0,11.0,70.0,592.0,472260.0



--- ESTADÍSTICA: POBLACIÓN ---


,count,mean,std,min,25%,50%,75%,max
Total,159.0,1.834382e+06,6.621906e+06,83052.0,325535.0,620637.0,1083091.5,49128297.0
periodo,159.0,2.023000e+03,8.190764e-01,2022.0,2022.0,2023.0,2024.0,2024.0


# **3. Análisis estructural de variables**

Se analizan los valores únicos, la cardinalidad y las categorías presentes en las variables, con el objetivo de comprender su distribución y detectar posibles necesidades de normalización.

In [5]:
#@title Valores únicos (nunique)
# #¿Qué hay dentro de mis datos?

# ============================================================
# FUNCIÓN: Perfilado dimensional de múltiples DataFrames
# Objetivo: Revisar estructura, tipos de datos y cardinalidad
# ============================================================

# Definimos una función llamada 'perfilado_dimensional'
# Recibe un parámetro llamado 'datasets', que debe ser un diccionario
# donde cada clave es un nombre y cada valor un DataFrame.
def perfilado_dimensional(datasets):
    """
    Realiza un escaneo de la estructura de los DataFrames para identificar
    la cardinalidad de cada variable antes del procesamiento.
    """

    # Recorremos el diccionario 'datasets'
    # nombre → clave del diccionario (string)
    # df → valor asociado (el DataFrame)
    for nombre, df in datasets.items():

        # ------------------------------------------------------------
        # Encabezado visual para separar el análisis de cada dataset
        # ------------------------------------------------------------
        print(f"\n" + "─" * 50)
        print(f" ESTRUCTURA DE DATOS: {nombre}")
        print("─" * 50)

        # ------------------------------------------------------------
        # Generamos el análisis de valores únicos (nunique)
        # Incluimos el tipo de dato para detectar fallos de formato
        # Creamos un DataFrame llamado 'analisis'
        # ------------------------------------------------------------
        analisis = pd.DataFrame({
            'Valores Únicos': df.nunique(),
            'Dtype': df.dtypes,
            'Ejemplo': [df[col].iloc[0] if len(df) > 0 else "N/A" for col in df.columns]
        })

        # ------------------------------------------------------------
        # Mostrar tabla resumen del análisis
        # ------------------------------------------------------------
        display(analisis)

        # ------------------------------------------------------------
        # Mostrar dimensiones del DataFrame (filas x columnas)
        # ------------------------------------------------------------
        print(f"Dimensiones actuales: {df.shape[0]} filas x {df.shape[1]} columnas")


# ============================================================
# DICCIONARIO DE DATASETS A ANALIZAR
# Cada clave es un nombre descriptivo y cada valor un DataFrame
# ============================================================

datasets = {
    "HECHOS_CONOCIDOS": df_hechos,
    "POBLACION_INE": df_poblacion
}

# ============================================================
# EJECUCIÓN DEL PERFILADO DIMENSIONAL
# ============================================================

perfilado_dimensional(datasets)


──────────────────────────────────────────────────
 ESTRUCTURA DE DATOS: HECHOS_CONOCIDOS
──────────────────────────────────────────────────


,Valores Únicos,Dtype,Ejemplo
Provincias,55,object,Total Nacional
Grupo penal,9,object,ACCESO E INTERCEPTACIÓN ILÍCITA
periodo,3,int64,2024
Total,661,int64,9773


Dimensiones actuales: 1485 filas x 4 columnas

──────────────────────────────────────────────────
 ESTRUCTURA DE DATOS: POBLACION_INE
──────────────────────────────────────────────────


,Valores Únicos,Dtype,Ejemplo
Provincias,53,object,Total Nacional
Total,159,int64,49128297
periodo,3,int64,2024


Dimensiones actuales: 159 filas x 3 columnas


In [6]:
#@title Análisis de Categorías por columnas
# ============================================================
# FUNCIÓN: Análisis detallado de categorías por columna
# Objetivo: Ver cuántas categorías tiene cada columna y listarlas
# ============================================================

def leyenda(df, columnas, nombre_dataset):
    # Encabezado visual para separar cada dataset
    print(f"\n{'='*60}")
    print(f" ESTRUCTURA DETALLADA: {nombre_dataset}")
    print(f"{'='*60}")

    # Estandarizamos nombres de columnas para evitar errores de ejecución
    cols_actuales = {c.lower(): c for c in df.columns}

    # Recorremos la lista de columnas que queremos analizar
    for col_buscada in columnas:

        # Buscamos la columna real en el DataFrame (ignorando mayúsculas)
        col_real = cols_actuales.get(col_buscada.lower())

        if col_real:
            # Extraemos valores únicos, eliminamos NaN y convertimos a string
            valores = sorted([str(v) for v in df[col_real].dropna().unique()])

            # Mostramos nombre de la columna y cuántas categorías tiene
            print(f"\n[+] Columna: '{col_real}' ({len(valores)} categorías)")
            print("-" * 40)

            # Si hay muchas categorías, mostramos un resumen inteligente
            if len(valores) > 20:
                for v in valores[:10]:
                    print(f"  • {v}")
                print(f"  ... (+ {len(valores)-15} categorías adicionales) ...")
                for v in valores[-5:]:
                    print(f"  • {v}")
            else:
                for v in valores:
                    print(f"  • {v}")

        else:
            # Mensaje de error si la columna no existe en el DataFrame
            print(f"\n[!] Error: La columna '{col_buscada}' no existe en {nombre_dataset}")


# ============================================================
# EJECUCIÓN DEL ANÁLISIS DE CATEGORÍAS
# ============================================================

# HECHOS CONOCIDOS
leyenda(
    df_hechos,
    ['Provincias', 'Grupo penal', 'periodo'],
    "HECHOS CONOCIDOS"
)

# POBLACIÓN INE
leyenda(
    df_poblacion,
    ['Provincias', 'periodo'],
    "POBLACIÓN INE"
)



 ESTRUCTURA DETALLADA: HECHOS CONOCIDOS

[+] Columna: 'Provincias' (55 categorías)
----------------------------------------
  • Albacete
  • Alicante/Alacant
  • Almería
  • Araba/Álava
  • Asturias
  • Badajoz
  • Balears (Illes)
  • Barcelona
  • Bizkaia
  • Burgos
  ... (+ 40 categorías adicionales) ...
  • Valencia/València
  • Valladolid
  • Zamora
  • Zaragoza
  • Ávila

[+] Columna: 'Grupo penal' (9 categorías)
----------------------------------------
  • ACCESO  E INTERCEPTACIÓN ILÍCITA
  • AMENAZAS Y COACCIONES
  • CONTRA EL HONOR
  • CONTRA LA PROPIEDAD INDUSTRIAL/INTELECTUAL
  • DELITOS SEXUALES
  • FALSIFICACIÓN INFORMÁTICA
  • FRAUDE INFORMÁTICO
  • INTERFERENCIA EN LOS DATOS Y EN EL SISTEMA
  • TOTAL grupo penal

[+] Columna: 'periodo' (3 categorías)
----------------------------------------
  • 2022
  • 2023
  • 2024

 ESTRUCTURA DETALLADA: POBLACIÓN INE

[+] Columna: 'Provincias' (53 categorías)
----------------------------------------
  • 01 Araba/Álava
  • 02 Albacete

## **4. Validación de coherencia entre datasets**

Se comparan las distintas fuentes de datos para asegurar la consistencia en variables clave como las provincias y otras dimensiones compartidas.

In [7]:
#@title Revisión Provincias en los dos archivos (Hechos vs INE)

# 1. PREPARACIÓN DE VECTORES DE COMPARACIÓN
# Limpiamos el INE de códigos numéricos y normalizamos espacios
ine_clean_names = df_poblacion['Provincias'].str.replace(r'^\d+\s', '', regex=True).str.strip()

# Extraemos nombres únicos de HECHOS
hechos_names = pd.Series(df_hechos['Provincias'].unique())

# 2. IDENTIFICACIÓN DE DISCORDANCIAS CROSS-DATASET
# Provincias que aparecen en HECHOS pero no en INE
faltantes = [p for p in hechos_names if p not in ine_clean_names.values]

# 3. ALGORITMO DE SUGERENCIA POR SIMILITUD (MAPPING LOGIC)
mapeo_sugerido = []
for p_hechos in faltantes:
    # Limpiamos el nombre de Hechos para la búsqueda (quitamos paréntesis y puntuación)
    clean_p_hechos = p_hechos.replace('(', '').replace(')', '').replace(',', '').strip()

    # Buscamos en el INE si el nombre de Hechos está contenido o viceversa
    coincidencia = [
        p_ine for p_ine in ine_clean_names.unique()
        if clean_p_hechos.split()[0] in p_ine or p_ine in clean_p_hechos
    ]

    mapeo_sugerido.append({
        'Valor en HECHOS': p_hechos,
        'Candidato en INE': coincidencia[0] if coincidencia else 'SIN COINCIDENCIA',
        'Estado': 'Pendiente de Ajuste' if coincidencia else 'Registro Excluido (N/A)'
    })

# Convertimos los resultados en un DataFrame
df_revision = pd.DataFrame(mapeo_sugerido)

# 4. RENDERIZADO DE TABLA DE CONTROL
print("--- TABLA DE CONTROL DE NOMENCLATURA (HECHOS vs INE) ---")
if not df_revision.empty:
    display(
        df_revision.style.set_properties(**{'text-align': 'left'})
        .set_table_styles([
            {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white')]}
        ])
    )
else:
    print("¡Perfecto! No se han detectado discrepancias de nombres entre HECHOS e INE.")


--- TABLA DE CONTROL DE NOMENCLATURA (HECHOS vs INE) ---


,Valor en HECHOS,Candidato en INE,Estado
0,Balears (Illes),"Balears, Illes",Pendiente de Ajuste
1,Coruña (A),"Coruña, A",Pendiente de Ajuste
2,Rioja (La),"Rioja, La",Pendiente de Ajuste
3,Palmas (Las),"Palmas, Las",Pendiente de Ajuste
4,En el extranjero,SIN COINCIDENCIA,Registro Excluido (N/A)
5,Desconocida,SIN COINCIDENCIA,Registro Excluido (N/A)


In [8]:
# @title Cuantificación Real: Desconocida / Extranjero
# Tras identificar las discrepancias en la Tabla de Control, se realiza una
# cuantificación tanto de registros (filas) como de volumen total para determinar
# el impacto real de excluir "Desconocida" y "En el extranjero".

# 1. DEFINICIÓN DE CATEGORÍAS BAJO ESTUDIO
categorias_especiales = ['Desconocida', 'En el extranjero']

# 2. ANÁLISIS EN HECHOS (Filtrando Totales de Grupo Penal)
filtro_hechos = (df_hechos['Grupo penal'] != 'TOTAL grupo penal')
df_h_esp = df_hechos[filtro_hechos & df_hechos['Provincias'].isin(categorias_especiales)]

hechos_stats = df_h_esp.groupby('Provincias')['Total'].agg(
    Filas_con_Datos = lambda x: (x > 0).sum(),
    Suma_Total_Hechos = 'sum'
)

# 3. RENDERIZADO DE RESULTADOS CON FORMATO
def fmt_milla(x):
    return "{:,.0f}".format(x).replace(",", ".")

print("--- AUDITORÍA DE DATOS EXCLUIDOS (CAPA EXTRA-PROVINCIAL) ---")
print("-" * 60)
print("ANÁLISIS DE HECHOS:")
display(hechos_stats.style.format(fmt_milla))

# 4. CÁLCULO DEL PESO RELATIVO (Opcional, para tu defensa)
total_h_sucio = df_hechos[df_hechos['Grupo penal'] != 'TOTAL grupo penal']['Total'].sum()
suma_h_excluida = hechos_stats['Suma_Total_Hechos'].sum()

print("-" * 60)
print(f"Impacto en Hechos: Se excluyen {fmt_milla(suma_h_excluida)} incidentes "
      f"({(suma_h_excluida/total_h_sucio)*100:.2f}% del total).")


--- AUDITORÍA DE DATOS EXCLUIDOS (CAPA EXTRA-PROVINCIAL) ---
------------------------------------------------------------
ANÁLISIS DE HECHOS:


,Filas_con_Datos,Suma_Total_Hechos
Provincias,,
Desconocida,0,0
En el extranjero,21,13.842


------------------------------------------------------------
Impacto en Hechos: Se excluyen 13.842 incidentes (0.53% del total).


#**5. Análisis de tipologías delictivas (Hechos)**
>
> En esta sección se estudia la distribución real de los hechos delictivos registrados por tipología penal, con el objetivo de identificar cuáles concentran el mayor volumen de denuncias y validar la coherencia interna del dataset.  
>
> La clasificación utilizada sigue la estructura oficial del **Sistema Estadístico de Criminalidad (SEC)** del Ministerio del Interior.
>
> **Referencia metodológica:** Se ha contrastado la correspondencia de los **9 Grupos Penales** con la tabla de tipologías incluida en la **página 4** del documento oficial [Metodología de Cibercriminalidad (SEC)](https://estadisticasdecriminalidad.ses.mir.es/publico/portalestadistico/dam/jcr:d96d4063-98d8-4647-8c76-d46a331a4ba3/03_Metodolog%C3%ADa_Cibercriminalidad.pdf), garantizando que la estructura del dataset coincide con la clasificación estandarizada.
>
> *Nota de control:* Se presta especial atención a la categoría **"Total grupo penal"**, que actúa como sumatorio global y no debe incluirse en el análisis estadístico, ya que podría generar duplicidades o inflar artificialmente el peso de determinadas tipologías.

In [9]:
#@title Ranking de Criminalidad (Hechos)

# ----------------------------------------------------------------------------
# FUNCIÓN DE APOYO: Limpieza de texto para comparaciones
# ----------------------------------------------------------------------------
def limpiar_texto_comparable(texto):
    """
    Limpia el texto quitando acentos, espacios extra y pasando a minúsculas
    para que los filtros de 'Total' no fallen por diferencias de formato.
    """
    if not isinstance(texto, str):
        return ""
    # Quitar acentos y normalizar
    texto = "".join(c for c in unicodedata.normalize('NFD', texto)
                    if unicodedata.category(c) != 'Mn')
    return texto.strip().lower()

# ----------------------------------------------------------------------------
# FUNCIÓN PRINCIPAL: Generar Ranking (HECHOS)
# ----------------------------------------------------------------------------
def generar_ranking_limpio(df, columna_valor, titulo_analisis):
    """
    Función para limpiar el dataset de las 'filas totales' del Ministerio
    y calcular el peso real de cada delito.
    """
    df_temp = df.copy()

    # ==========================================================================
    # 1. ELIMINACIÓN DE CAPAS (AUDITORÍA DE INTEGRIDAD)
    # ==========================================================================

    df_temp['Tipologia_Limpia'] = df_temp['Grupo penal'].apply(limpiar_texto_comparable)

    # Términos que identifican 'filas de sumatorio' y no datos atómicos
    #se elimina del ranking desconocida, en el extranjero así como filas de Totales
    terminos_prohibidos = ['total grupo penal', 'total nacional']

    condicion_limpieza = (
        (~df_temp['Tipologia_Limpia'].isin(terminos_prohibidos)) &
        (~df_temp['Provincias'].str.contains('Total Nacional', case=False)) &
        (~df_temp['Provincias'].str.contains('Desconocida', case=False)) &
        (~df_temp['Provincias'].str.contains('En el extranjero', case=False))
)

    # Aplicamos la limpieza y trabajamos sobre una copia segura
    df_limpio = df_temp[condicion_limpieza].copy()

    # ==========================================================================
    # 2. PROCESAMIENTO ESTADÍSTICO
    # ==========================================================================

    # Ponemos los nombres bonitos (Ej: FRAUDE -> Fraude)
    df_limpio['Grupo penal'] = df_limpio['Grupo penal'].str.strip().str.title()

    ranking = df_limpio.groupby('Grupo penal')[columna_valor].sum().sort_values(ascending=False).reset_index()

    # Cálculo de porcentajes para análisis de Pareto
    total_real = ranking[columna_valor].sum()
    ranking['% Individual'] = (ranking[columna_valor] / total_real) * 100
    ranking['% Acumulado'] = ranking['% Individual'].cumsum()

    # ==========================================================================
    # 3. ESTÉTICA Y FORMATO
    # ==========================================================================

    color_barra = '#3498db'  # Color para HECHOS

    fmt_miles = lambda x: "{:,.0f}".format(x).replace(",", ".")

    estilo = ranking.style.format({
        columna_valor: fmt_miles,
        '% Individual': '{:.2f}%',
        '% Acumulado': '{:.2f}%'
    }).bar(subset=['% Individual'], color=color_barra) \
      .set_caption(f"RANKING DEPURADO: {titulo_analisis}") \
      .set_table_styles([
          {'selector': 'th', 'props': [('background-color', '#2c3e50'), ('color', 'white'), ('text-align', 'center')]},
          {'selector': 'td.col0', 'props': [('text-align', 'left')]}
      ])

    # Resumen de auditoría por pantalla
    print(f"\n✅ {titulo_analisis}: Tipologías procesadas correctamente.")
    print(f"📊 Volumen real depurado (Capa Atómica): {fmt_miles(total_real)} registros.")
    display(estilo)

# ----------------------------------------------------------------------------
# Ejecución final (SOLO HECHOS)
# ----------------------------------------------------------------------------
generar_ranking_limpio(df_hechos, 'Total', "Hechos Conocidos (Denuncias)")



✅ Hechos Conocidos (Denuncias): Tipologías procesadas correctamente.
📊 Volumen real depurado (Capa Atómica): 1.297.956 registros.


,Grupo penal,Total,% Individual,% Acumulado
0,Fraude Informático,1.163.149,89.61%,89.61%
1,Amenazas Y Coacciones,50.889,3.92%,93.53%
2,Falsificación Informática,47.067,3.63%,97.16%
3,Acceso E Interceptación Ilícita,22.536,1.74%,98.90%
4,Interferencia En Los Datos Y En El Sistema,5.274,0.41%,99.30%
5,Delitos Sexuales,5.096,0.39%,99.70%
6,Contra El Honor,3.487,0.27%,99.96%
7,Contra La Propiedad Industrial/Intelectual,458,0.04%,100.00%


# **Leyenda de Correspondencias (Clustering) Hechos**

Para el análisis, se mantienen las **8 categorías originales** del Sistema Estadístico de Criminalidad (SEC), ya que estas ya representan un nivel de agregación técnica que permite comparar el volumen de Hechos de forma estandarizada.

---

| Grupo Penal (en Dataset) | Hechos Específicos que engloba (según SEC) |
| :--- | :--- |
| **Fraude Informático** | Estafas bancarias, estafas con tarjetas, cheques, inversiones y otras estafas informáticas. |
| **Amenazas y Coacciones** | Amenazas, coacciones, extorsiones y acoso a través de medios digitales. |
| **Falsificación Informática** | Usurpación de estado civil y falsificación de documentos o tarjetas de crédito/débito. |
| **Acceso e Interceptación** | Descubrimiento y revelación de secretos y acceso ilegal a sistemas informáticos. |
| **Delitos Sexuales** | Grooming, sexting, pornografía infantil y otros delitos de libertad sexual en red. |
| **Interferencia (Datos/Sistema)** | Daños informáticos, borrado de datos y ataques contra la estabilidad de sistemas. |
| **Contra el Honor** | Injurias y calumnias vertidas a través de redes sociales o medios telemáticos. |
| **Propiedad Ind. / Intelectual** | Delitos contra la propiedad intelectual y la propiedad industrial en el entorno ciber. |

# **Resumen de Hallazgos: Auditoría de Datos (2022-2024)**

Tras analizar las fuentes del Ministerio del Interior y del INE, se han extraído las siguientes conclusiones clave antes de proceder al análisis visual:

* **Identificación del volumen real:** Se ha detectado que los datos originales contienen filas de sumatorios (Totales). Al filtrar estas capas y dejar exclusivamente la **capa atómica** (provincias y grupos específicos), el volumen real se sitúa en **1.297.956 incidentes**.
* **Predominio del Fraude:** La cibercriminalidad en España tiene un motor claro: el **Fraude Informático**, que representa el **89,61%** de la actividad delictiva analizada en este periodo.
* **Impacto de Exclusiones:** La eliminación de registros "En el extranjero" y "Desconocida" solo supone una pérdida del **0,53%** del total de hechos, lo que garantiza que el modelo final será altamente representativo de la realidad nacional.
* **Sincronización de Fuentes:** Se ha verificado que ambos datasets están perfectamente alineados en el periodo **2022-2024**.
* **Necesidad de Normalización:** Se han identificado **4 provincias** (Balears, Coruña, Rioja y Palmas) con discrepancias de puntuación y el uso de códigos numéricos en el INE, lo que requiere un ajuste de etiquetas previo al cruce de datos (*merge*).

**En definitiva:** Los datos procesados son de alta calidad y ofrecen una base estadística sólida. Una vez unificada la nomenclatura de las provincias, el dataset estará listo para el cálculo de tasas y la generación de visualizaciones.

# **6. Pipeline de Limpieza y Consolidación (Dataset Maestro Multidimensional)**

El objetivo central de este pipeline es transformar los dos datasets independientes en una **única fuente de verdad basada en Hechos Conocidos**. La arquitectura se ha rediseñado para ofrecer una **jerarquía territorial dual (Provincia/CCAA)**, permitiendo granularidad en el modelado predictivo y claridad en el análisis visual.

**Fase 1: Filtrado de la Capa Atómica y Limpieza de Columnas**
Para garantizar la integridad y centrar el estudio en los hechos denunciados:
* **Filtrado de Registros No Válidos:** Se excluyen las filas de agregación para evitar la "doble contabilidad":
    * **Agregados:** Eliminación de `Total Nacional` y `TOTAL grupo penal`.
    * **Depuración Territorial:** Se eliminan `Desconocida` y `En el extranjero` para acotar el modelo a las 52 unidades provinciales.
* **Volumen de Control:** Este proceso asegura trabajar sobre la base validada de **1.297.956 hechos**, cifra confirmada en el Ranking de Criminalidad (2022-2024).

**Fase 2: Normalización y Enriquecimiento Territorial (Mapping)**
* **Normalización de Provincias:** Uso del transformador `ProvinciaNormalizer` para estandarizar nombres a formato *snake_case* (ej. `santa_cruz_de_tenerife`), eliminando ruidos de codificación y corrigiendo las 4 discrepancias detectadas (Balears, Coruña, Rioja y Palmas).
* **Creación de la Dimensión CCAA:** Se incorpora una nueva columna de **Comunidad Autónoma** mediante un mapeo semántico de las 52 provincias. Esto permite realizar agrupaciones en el EDA Gráfico y mitigar la dispersión visual de los datos (52 provincias).
* **Cruce Maestro:** Se realiza un *merge* de las 2 fuentes (Hechos y Población) utilizando como claves: `Provincias` y `periodo`.

**Fase 3: Ingeniería de Variables y Métricas (Feature Engineering)**
Se definen las métricas que describen la actividad criminal:
1.  **Formateo de Grupos Penales:** Normalización sintáctica para facilitar la creación de etiquetas en algoritmos de Machine Learning.
2.  **Cálculo de Métricas de Intensidad y Tasa:**
    * **Total_Hechos:** Volumen de denuncias (Variable predictora principal).
    * **Tasa_Criminalidad:** `(Total_Hechos / Poblacion) * 1.000` (**Variable Objetivo / Target**), ajustada a la escala estándar del Ministerio para grandes volúmenes de delitos.

**Fase 4: Exportación y Preparación para el EDA Gráfico**
Generación del archivo **`df_cibercrimen.csv`**. El dataset resultante queda estructurado como una serie temporal enriquecida, lista para el análisis de tendencias en el **Notebook 03**.

In [10]:
#@title 6.1 Pipeline Final: Consolidación, Formateo, creación CCAA y Auditoría (2022-2024)

# ----------------------------------------------------------------------------
# 1. CLASE DE NORMALIZACIÓN (Arquitectura Reutilizable)
# ----------------------------------------------------------------------------
class ProvinciaNormalizer(BaseEstimator, TransformerMixin):
    """
    OBJETIVO: Estandarizar nombres de provincias (ej: '01 Araba/Álava' -> 'araba')
    para que el cruce entre las tablas sea perfecto.
    """
    def fit(self, X, y=None):
        return self

    def _norm_string(self, text):
        if not isinstance(text, str): return ""
        text = re.sub(r"^\d{2}\s+", "", text)  # Quitar códigos INE
        text = "".join(c for c in unicodedata.normalize('NFD', text)
                       if unicodedata.category(c) != 'Mn')  # Quitar acentos
        text = re.split(r'[/|(|,]', text)[0].strip().lower()  # Limpiar barras/paréntesis
        return text.replace(" ", "_")

    def transform(self, X):
        df = X.copy()
        if "Provincias" in df.columns:
            df["Provincias"] = df["Provincias"].apply(self._norm_string)
        return df


# ----------------------------------------------------------------------------
# 2. FUNCIÓN PRINCIPAL DEL PIPELINE (AJUSTADA A METODOLOGÍA 2022-2024)
# ----------------------------------------------------------------------------
def pipeline_limpieza_hechos(df_h, df_p):
    """
    Motor de unificación: Une Hechos + Población + CCAA en una sola tabla.
    """

    # --- 1. MAPEO CCAA ---
    mapeo_completo_ccaa = {
        'albacete': 'Castilla-La Mancha', 'ciudad_real': 'Castilla-La Mancha', 'cuenca': 'Castilla-La Mancha',
        'guadalajara': 'Castilla-La Mancha', 'toledo': 'Castilla-La Mancha',
        'avila': 'Castilla y León', 'burgos': 'Castilla y León', 'leon': 'Castilla y León',
        'palencia': 'Castilla y León', 'salamanca': 'Castilla y León', 'segovia': 'Castilla y León',
        'soria': 'Castilla y León', 'valladolid': 'Castilla y León', 'zamora': 'Castilla y León',
        'almeria': 'Andalucía', 'cadiz': 'Andalucía', 'cordoba': 'Andalucía', 'granada': 'Andalucía',
        'huelva': 'Andalucía', 'jaen': 'Andalucía', 'malaga': 'Andalucía', 'sevilla': 'Andalucía',
        'huesca': 'Aragón', 'teruel': 'Aragón', 'zaragoza': 'Aragón',
        'asturias': 'Asturias', 'balears': 'Islas Baleares',
        'palmas': 'Canarias', 'santa_cruz_de_tenerife': 'Canarias',
        'cantabria': 'Cantabria',
        'barcelona': 'Cataluña', 'girona': 'Cataluña', 'lleida': 'Cataluña', 'tarragona': 'Cataluña',
        'alicante': 'C. Valenciana', 'castellon': 'C. Valenciana', 'valencia': 'C. Valenciana',
        'badajoz': 'Extremadura', 'caceres': 'Extremadura',
        'coruna': 'Galicia', 'lugo': 'Galicia', 'ourense': 'Galicia', 'pontevedra': 'Galicia',
        'madrid': 'Madrid', 'murcia': 'Murcia', 'navarra': 'Navarra',
        'araba': 'País Vasco', 'bizkaia': 'País Vasco', 'gipuzkoa': 'País Vasco',
        'rioja': 'La Rioja', 'ceuta': 'Ceuta', 'melilla': 'Melilla'
    }

    # --- 2. FILTRADO ---
    def filtrar_basicos(df):
        df = df.copy()
        mask = (
            ~df['Provincias'].str.contains('Total Nacional', case=False) &
            ~df['Provincias'].str.contains('Desconocida', case=False) &
            ~df['Provincias'].str.contains('En el extranjero', case=False)
        )
        if 'Grupo penal' in df.columns:
            mask &= (df['Grupo penal'] != 'TOTAL grupo penal')
        return df[mask]

    df_h_atomo = filtrar_basicos(df_h)
    df_p_atomo = filtrar_basicos(df_p)

    # --- 3. NORMALIZACIÓN ---
    norm = ProvinciaNormalizer()
    df_h_atomo = norm.transform(df_h_atomo)
    df_p_atomo = norm.transform(df_p_atomo)

    # Normalizar delitos
    df_h_atomo['Grupo penal'] = (
        df_h_atomo['Grupo penal']
        .str.lower()
        .str.normalize('NFD')
        .str.replace(r'[\u0300-\u036f]', '', regex=True)
        .str.replace(" ", "_")
    )

    # --- 4. AGREGACIÓN POBLACIÓN ---
    df_p_agrupado = df_p_atomo.groupby(['Provincias', 'periodo'], as_index=False)['Total'].sum()
    df_p_agrupado.rename(columns={'Total': 'Poblacion_Provincial'}, inplace=True)

    # --- 5. MERGE FINAL ---
    df_master = pd.merge(df_h_atomo, df_p_agrupado,
                         on=['Provincias', 'periodo'], how='left')
    df_master.rename(columns={'Total': 'Total_Hechos'}, inplace=True)

    # --- 6. CREACIÓN DE CCAA ---
    df_master['CCAA'] = df_master['Provincias'].map(mapeo_completo_ccaa)

    # --- 7. MÉTRICAS (CORREGIDO A X 1.000 SEGÚN SEC) ---
    df_master['Tasa_Criminalidad'] = (
        df_master['Total_Hechos'] / df_master['Poblacion_Provincial']
    ) * 1000

    df_master.fillna(0, inplace=True)

    return df_master


# ----------------------------------------------------------------------------
# 3. EJECUCIÓN, FORMATEO Y AUDITORÍA
# ----------------------------------------------------------------------------

df_cibercrimen = pipeline_limpieza_hechos(df_hechos, df_poblacion)

# Redondeo final
df_cibercrimen['Tasa_Criminalidad'] = df_cibercrimen['Tasa_Criminalidad'].round(4)

print(f"✅ Dataset Maestro creado con éxito (2022-2024). Filas: {len(df_cibercrimen)}")
display(df_cibercrimen.head())

print("\n" + "="*50)
print("📊 ESTRUCTURA TÉCNICA DEL DATASET (df.info)")
print("="*50)
df_cibercrimen.info()

print("\n" + "="*50)
print("🔍 AUDITORÍA FINAL DE VOLUMEN (Capa Atómica)")
print("="*50)
sum_hechos = df_cibercrimen['Total_Hechos'].sum()
print(f"🔹 Suma Total de HECHOS: {sum_hechos:,.0f}".replace(",", "."))

print("\n--- ESTADÍSTICA DESCRIPTIVA GLOBAL ---")
display(df_cibercrimen.describe().T)

# Exportación final
ruta_salida = os.path.join(path_data, "df_cibercrimen.csv")
df_cibercrimen.to_csv(ruta_salida, index=False, sep=';')
print(f"✅ Archivo actualizado en: {ruta_salida}")
print("\n💾 Archivo 'df_cibercrimen.csv' generado. Todo listo para Notebook 03.")

✅ Dataset Maestro creado con éxito (2022-2024). Filas: 1248


,Provincias,Grupo penal,periodo,Total_Hechos,Poblacion_Provincial,CCAA,Tasa_Criminalidad
0,araba,acceso__e_interceptacion_ilicita,2024,32,341961,País Vasco,0.0936
1,araba,acceso__e_interceptacion_ilicita,2023,49,338594,País Vasco,0.1447
2,araba,acceso__e_interceptacion_ilicita,2022,38,336308,País Vasco,0.1130
3,araba,amenazas_y_coacciones,2024,236,341961,País Vasco,0.6901
4,araba,amenazas_y_coacciones,2023,232,338594,País Vasco,0.6852



📊 ESTRUCTURA TÉCNICA DEL DATASET (df.info)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1248 entries, 0 to 1247
Data columns (total 7 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Provincias            1248 non-null   object 
 1   Grupo penal           1248 non-null   object 
 2   periodo               1248 non-null   int64  
 3   Total_Hechos          1248 non-null   int64  
 4   Poblacion_Provincial  1248 non-null   int64  
 5   CCAA                  1248 non-null   object 
 6   Tasa_Criminalidad     1248 non-null   float64
dtypes: float64(1), int64(3), object(3)
memory usage: 68.4+ KB

🔍 AUDITORÍA FINAL DE VOLUMEN (Capa Atómica)
🔹 Suma Total de HECHOS: 1.297.956

--- ESTADÍSTICA DESCRIPTIVA GLOBAL ---


,count,mean,std,min,25%,50%,75%,max
periodo,1248.0,2023.000000,8.168239e-01,2022.0,2022.0000,2023.0000,2.024000e+03,2.024000e+03
Total_Hechos,1248.0,1040.028846,4.608321e+03,0.0,9.0000,49.0000,2.512500e+02,7.153400e+04
Poblacion_Provincial,1248.0,934829.185897,1.233782e+06,83052.0,324852.7500,618599.0000,1.028139e+06,7.113886e+06
Tasa_Criminalidad,1248.0,1.088506,2.588476e+00,0.0,0.0236,0.0731,3.695250e-01,1.243440e+01


✅ Archivo actualizado en: ..\01_Data\df_cibercrimen.csv

💾 Archivo 'df_cibercrimen.csv' generado. Todo listo para Notebook 03.
